<a href="https://colab.research.google.com/github/dannroldan/Procesos-estocasticos/blob/main/matriz_fundamental.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Problema 1: Serpientes y Escaleras

## Solución analítica

Definimos el espacio de estados $S = \{0, 1, 2, \dots, 20\}$, donde $0$ es la posición de empezar (fuera del tablero) y $20$ es el estado absorbente final (la meta).

**Reglas para este modelo:**
1. Se avanza mediante un dado normal de 6 caras.
2. Para ganar, el jugador necesita ya sea alcanzar o superar la casilla 20 (cualquier tirada que resulte en $X_{t+1} \ge 20$ se considera como a 20).
3. Las transiciones del tablero son:
   * **Escaleras:** $3 \to 11$, $15 \to 19$
   * **Serpientes:** $13 \to 4$, $17 \to 10$

Sea $P$ la matriz de transición de tamaño $21 \times 21$. La probabilidad de transición del estado $i$ al estado $j$ se define como:

$$P_{i,j} = \frac{1}{6} \sum_{k=1}^{6} \mathbb{1}(D(i+k) = j)$$

donde $D(x)$ es la función destino que aplica las reglas del tablero, y $\mathbb{1}$ es la función indicadora.

Para encontrar el número promedio de tiradas (i.e. tiempo esperado de absorción), particionamos la matriz $P$ para aislar los estados transitorios (0 al 19).
Sea $Q$ la submatriz de transiciones entre estados transitorios. La matriz fundamental $N$ se define como:

$$N = (I - Q)^{-1}$$

El vector de tiempos esperados $\mathbf{t}$ desde cada estado transitorio es:

$$\mathbf{t} = N \mathbf{1}$$

donde $\mathbf{1}$ es un vector columna de unos. El valor de $\mathbf{t}_0$ nos dará el promedio de tiradas desde la casilla de inicio.

In [ ]:
import numpy as np

#sol analítica
saltos = {3: 11, 13: 4, 15: 19, 17: 10}

def destino(casilla):
    if casilla >= 20:
        return 20
    return saltos.get(casilla, casilla)

P = np.zeros((21, 21))
for i in range(21):
    if i == 20:
        P[i, i] = 1.0
    else:
        for dado in range(1, 7):
            j = destino(i + dado)
            P[i, j] += 1/6

Q = P[:20, :20]
I = np.eye(20)
N = np.linalg.inv(I - Q)
t = N @ np.ones(20)

esperanza_analitica = t[0]

#monte caflo
def simular_juego():
    estado = 0
    tiradas = 0
    while estado < 20:
        dado = np.random.randint(1, 7)
        estado = destino(estado + dado)
        tiradas += 1
    return tiradas

n_simulaciones = 100000
resultados_mc = [simular_juego() for _ in range(n_simulaciones)]
esperanza_simulada = np.mean(resultados_mc)

print(f"Núm promedio de tiradas (analítico): {esperanza_analitica:.4f}")
print(f"Núm promedio de tiradas (simulación MC): {esperanza_simulada:.4f}")

Número promedio de tiradas (Analítico): 7.0635
Número promedio de tiradas (Simulación Monte Carlo): 7.0478


# Problema 2: El Laberinto del Ratón

## Solución analítica

El laberinto se modela tomando cada habitación como un nodo y los espacios sin paredes son tipo aristas. El ratón elige cualquier salida desde su habitación con probabilidad uniforme de las que sí se pueden.

Las conexiones que tendríamos son:
* $0: \{1, 2\}$
* $1: \{0, 3, 7\}$
* $2: \{0, 3, 8\}$
* $3: \{1, 2, 4, 5\}$
* $4: \{3, 6, 7\}$
* $5: \{3, 6, 8\}$
* $6: \{4, 5\}$
* $7$ (comida) y $8$ (shock) son estados absorbentes.


Sea $u_i$ la probabilidad de alcanzar el estado $7$ antes que el $8$ partiendo del estado $i$:
$$u_7 = 1, \quad u_8 = 0$$
$$u_i = \sum_{j} p_{i,j} u_j \quad \text{para todo estado transitorio } i$$

Particionamos la matriz de transición:
$$P = \begin{pmatrix} Q & R \\ 0 & I \end{pmatrix}$$

La matriz de probabilidades de absorción $B$ se obtiene mediante:
$$B = (I - Q)^{-1} R$$

Donde $B_{i,j}$ representa la probabilidad de ser absorbido por el estado $j$ empezando en el estado transitorio $i$.

In [ ]:
import numpy as np

#sol analítica
adj = {
    0: [1, 2],
    1: [0, 3, 7],
    2: [0, 3, 8],
    3: [1, 2, 4, 5],
    4: [3, 6, 7],
    5: [8, 3, 6],
    6: [5, 4]
}

Q_maze = np.zeros((7, 7))
R_maze = np.zeros((7, 2))

for i in range(7):
    grados = len(adj[i])
    for j in adj[i]:
        if j == 7:
            R_maze[i, 0] = 1 / grados
        elif j == 8:
            R_maze[i, 1] = 1 / grados
        else:
            Q_maze[i, j] = 1 / grados

I_maze = np.eye(7)
N_maze = np.linalg.inv(I_maze - Q_maze)
B_maze = N_maze @ R_maze

prob_analitica = B_maze[0, 0]

#simulación mc
def simular_raton():
    estado = 0
    while estado not in [7, 8]:
        estado = np.random.choice(adj[estado])
    return 1 if estado == 7 else 0

n_simulaciones_raton = 100000
exitos = sum(simular_raton() for _ in range(n_simulaciones_raton))
prob_simulada = exitos / n_simulaciones_raton

print(f"Prob de alcanzar la comida (analítica): {prob_analitica:.4f}")
print(f"Prob de alcanzar la comida (simulación): {prob_simulada:.4f}")

Prob de alcanzar la comida (analítica): 0.5000
Prob de alcanzar la comida (simulación): 0.4957
